# English inference (generate, listen, final results): CPU is enough

Runtime: **CPU**. No GPU needed. Needs `inference.py` in the `tts_project` folder on Drive.

| Cell | What it does |
|---|---|
| 1 | Rebuilds the repo and config from Drive (a few minutes on a fresh runtime) |
| 2 | Generates E1, E2a, E2b, E3, E4 for each checkpoint in `STEPS` |
| 3 | Plays one sentence from every version, and prints the phoneme sequences |
| 4 | Writes the FINAL results to `Results/English` for the chosen checkpoint |

## Cell 1: setup

In [1]:
import os
os.chdir("/content")
from google.colab import drive
drive.mount('/content/drive')
import json, shutil, glob, re
D = "/content/drive/MyDrive/tts_project"
!pip install -q pyworld pypinyin unidecode inflect librosa pyyaml tgt matplotlib scikit-learn g2p_en num2words
import nltk
for r in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "punkt", "punkt_tab", "cmudict"]:
    nltk.download(r, quiet=True)
if not os.path.isdir("/content/FastSpeech2/text"):
    shutil.rmtree("/content/FastSpeech2", ignore_errors=True)
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "repo patches applied"
    !cp -r {D}/config_Svarah /content/FastSpeech2/config/Svarah
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
P = "/content/FastSpeech2/preprocessed_data/Svarah"
if not os.path.exists(f"{P}/stats.json"):
    !mkdir -p /content/FastSpeech2/preprocessed_data
    !tar -xf {D}/preprocessed_Svarah.tar -C /content/FastSpeech2/preprocessed_data Svarah/speakers.json Svarah/stats.json
os.chdir("/content/FastSpeech2")
assert os.path.exists(f"{D}/inference.py"), "inference.py is missing from the tts_project folder on Drive"
shutil.copy(f"{D}/inference.py", "inference.py")
spk = json.load(open(f"{P}/speakers.json"))
print("ready. speakers:", len(spk), "| S000 id:", spk.get("S000"))
print("checkpoints on Drive:", sorted(int(re.findall(r"(\d+)\.pth\.tar$", p)[0]) for p in glob.glob(f"{D}/output/ckpt/Svarah/*.pth.tar")))

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
repo patches applied
ready. speakers: 60 | S000 id: 48
checkpoints on Drive: [3000, 6000, 9000, 12000, 15000]


## Cell 2: generate (edit STEPS and SPEAKER)

In [2]:
# EDIT THESE TWO LINES, then run (CPU is enough)
STEPS = [9000, 15000]      # checkpoints to generate and compare
SPEAKER = "S000"           # speaker with the most training data
OUT = f"{D}/Results/English_test"
os.chdir("/content/FastSpeech2")
for step in STEPS:
    print("=== checkpoint", step, "| speaker", SPEAKER)
    !python inference.py --lang english --dataset Svarah --restore_step {step} --speaker {SPEAKER} --out {OUT}/step_{step}_{SPEAKER} 2>&1 | tail -8

=== checkpoint 9000 | speaker S000
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Removing weight norm...
E1: 87 phones, 5.18 s of audio
E2a: 51 phones, 3.26 s of audio
E2b: 56 phones, 3.34 s of audio
E3: 37 phones, 2.45 s of audio
E4: 81 phones, 5.64 s of audio
=== checkpoint 15000 | speaker S000
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Removing weight norm...
E1: 87 phones, 5.14 s of audio
E2a: 51 phones, 3.12 s of audio
E2b: 56 phones, 3.47 s of audio
E3: 37 phones, 2.43 s of audio
E4: 81 phones, 5.56 s of audio


## Cell 3: listen and compare

In [3]:
# Listen and compare the same sentence across every version generated so far
from IPython.display import Audio, display
SENTENCE = "E1"       # E1, E2a, E2b, E3 or E4
for r in sorted(glob.glob(f"{D}/Results/English_test/step_*")):
    f = f"{r}/test_generated_audios/{SENTENCE}.wav"
    if os.path.exists(f):
        print("---", os.path.basename(r)); display(Audio(f))
print(open(sorted(glob.glob(f"{D}/Results/English_test/step_*"))[-1] + "/phoneme_sequences.txt", encoding="utf-8").read())

--- step_15000_S000


--- step_9000_S000


E1
TEXT: The antidisestablishmentarianism movement gained considerable momentum during the parliamentary proceedings.
NORMALISED: the antidisestablishmentarianism movement gained considerable momentum during the parliamentary proceedings.
PHONES: DH AH0 AE2 N T AY0 D IH0 S AH0 S T AE2 B L IH0 SH M AH0 N T EH1 R IY0 AH0 N IH0 Z AH0 M M UW1 V M AH0 N T G EY1 N D K AH0 N S IH1 D ER0 AH0 B AH0 L M OW0 M EH1 N T AH0 M D UH1 R IH0 NG DH AH0 P AA2 R L AH0 M EH1 N T ER0 IY0 P R OW0 S IY1 D IH0 NG Z

E2a
TEXT: The orchestra's conduct during the ceremony was praised by the audience.
NORMALISED: the orchestra's conduct during the ceremony was praised by the audience.
PHONES: DH AH0 AO1 R K AH0 S T R AH0 Z K AA1 N D AH0 K T D UH1 R IH0 NG DH AH0 S EH1 R AH0 M OW2 N IY0 W AA1 Z P R EY1 Z D B AY1 DH AH0 AA1 D IY0 AH0 N S

E2b
TEXT: The researchers will conduct an experiment to evaluate the proposed method.
NORMALISED: the researchers will conduct an experiment to evaluate the proposed method.
PHONES

## Cell 4: final results folder

In [4]:
# FINAL results in the folder the assignment requires. Set STEP and SPEAKER to your chosen version.
STEP, SPEAKER = 9000, "S000"
os.chdir("/content/FastSpeech2")
!python inference.py --lang english --dataset Svarah --restore_step {STEP} --speaker {SPEAKER} --out {D}/Results/English 2>&1 | tail -8
!ls {D}/Results/English {D}/Results/English/test_generated_audios {D}/Results/English/variance_adaptor_plots

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Removing weight norm...
E1: 87 phones, 5.18 s of audio
E2a: 51 phones, 3.26 s of audio
E2b: 56 phones, 3.34 s of audio
E3: 37 phones, 2.45 s of audio
E4: 81 phones, 5.64 s of audio
/content/drive/MyDrive/tts_project/Results/English:
mel_plots	       training_loss.png       variance_data
phoneme_sequences.txt  validation_loss.png
test_generated_audios  variance_adaptor_plots

/content/drive/MyDrive/tts_project/Results/English/test_generated_audios:
E1.wav	E2a.wav  E2b.wav  E3.wav  E4.wav

/content/drive/MyDrive/tts_project/Results/English/variance_adaptor_plots:
duration  energy  pitch
